# 神经风格迁移 (Neural Style Transfer)

## 项目简介

本项目实现了经典的**神经风格迁移**算法，基于 Gatys 等人 2015 年的论文 *"A Neural Algorithm of Artistic Style"*。

### 核心思想
- 给定一张**内容图像**（Content Image）和一张**风格图像**（Style Image）
- 算法会生成一张新图像，保留内容图像的**语义结构**，同时融合风格图像的**艺术风格**（纹理、色彩、笔触等）

### 技术原理
- 使用预训练的 **VGG19** 网络提取图像特征
- **内容损失**：通过比较生成图像与内容图像在深层网络中的特征表示来保留内容
- **风格损失**：通过比较生成图像与风格图像的 **Gram 矩阵**（特征相关性矩阵）来迁移风格
- 使用 **L-BFGS** 优化器迭代优化生成图像，使总损失最小化

## 第一步：导入必要的库

In [ ]:
import torch                                    # PyTorch 深度学习框架
import torch.nn as nn                           # 神经网络模块
import torch.optim as optim                     # 优化器模块
import torchvision.transforms as transforms     # 图像变换工具
import torchvision.models as models             # 预训练模型库
from PIL import Image                           # 图像读取与处理
import matplotlib.pyplot as plt                 # 图像可视化
import copy                                     # 深拷贝模块，用于复制模型

# 打印 PyTorch 版本，确认环境正常
print(f"PyTorch 版本: {torch.__version__}")

## 第二步：配置计算设备

风格迁移是计算密集型任务，优先使用 GPU 加速。如果没有 GPU，则回退到 CPU。

In [ ]:
# 检测是否有可用的 CUDA GPU，有则使用 GPU，否则使用 CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 根据设备类型设置图像大小：GPU 内存充足时使用较大尺寸，CPU 时使用较小尺寸以加快速度
imsize = 512 if torch.cuda.is_available() else 256

print(f"使用设备: {device}")
print(f"图像尺寸: {imsize}x{imsize}")

## 第三步：定义图像预处理与后处理函数

- **预处理**：将图像调整大小、转为张量、并进行标准化（与 VGG19 训练时的预处理一致）
- **后处理**：将网络输出的张量转回可显示的图像格式

In [ ]:
# 定义图像预处理流水线
loader = transforms.Compose([
    transforms.Resize((imsize, imsize)),  # 将图像缩放到统一尺寸
    transforms.ToTensor(),                # 将 PIL 图像转换为 [0,1] 范围的张量
])

def load_image(image_path):
    """
    加载图像并进行预处理。
    
    参数:
        image_path: 图像文件路径
    返回:
        预处理后的图像张量，形状为 [1, C, H, W]
    """
    image = Image.open(image_path).convert("RGB")  # 打开图像并转为 RGB 格式
    image = loader(image)                           # 应用预处理变换
    image = image.unsqueeze(0)                      # 增加 batch 维度: [C,H,W] -> [1,C,H,W]
    return image.to(device)                         # 将张量移动到计算设备上


def tensor_to_image(tensor):
    """
    将张量转换回 PIL 图像，用于显示。
    
    参数:
        tensor: 形状为 [1, C, H, W] 的图像张量
    返回:
        PIL.Image 对象
    """
    image = tensor.cpu().clone()          # 将张量复制到 CPU
    image = image.squeeze(0)              # 移除 batch 维度: [1,C,H,W] -> [C,H,W]
    image = image.clamp(0, 1)             # 将像素值裁剪到 [0,1] 范围内
    image = transforms.ToPILImage()(image) # 将张量转换为 PIL 图像
    return image


print("图像处理函数定义完成 ✓")

## 第四步：加载内容图像和风格图像

请将你的图像放入 `images/` 目录下：
- `images/content.jpg` — 内容图像（你想要保留结构的照片）
- `images/style.jpg` — 风格图像（你想要迁移风格的艺术画）

> **提示**：如果没有准备图像，下面的代码会自动生成示例图像用于演示。

In [ ]:
import os  # 文件路径操作

# 定义图像路径
content_image_path = "images/content.jpg"  # 内容图像路径
style_image_path = "images/style.jpg"      # 风格图像路径

# 确保 images 目录存在
os.makedirs("images", exist_ok=True)

# 如果图像文件不存在，自动生成示例图像用于演示
if not os.path.exists(content_image_path):
    print("未找到内容图像，正在生成示例图像...")
    import numpy as np
    # 生成一个带有简单几何形状的示例内容图像
    img = np.zeros((512, 512, 3), dtype=np.uint8)        # 创建黑色背景
    img[100:400, 100:400] = [135, 206, 235]              # 中间放一个天蓝色方块
    img[150:350, 200:350] = [255, 165, 0]                # 叠加一个橙色方块
    img[200:300, 150:250] = [34, 139, 34]                # 再叠加一个绿色方块
    # 添加渐变背景
    for i in range(512):
        for j in range(512):
            if img[i, j].sum() == 0:                     # 仅修改黑色背景像素
                img[i, j] = [int(i/2), int(j/2), 128]    # 用位置生成渐变色
    Image.fromarray(img).save(content_image_path)         # 保存为 JPEG 文件
    print(f"示例内容图像已保存到: {content_image_path}")

if not os.path.exists(style_image_path):
    print("未找到风格图像，正在生成示例图像...")
    import numpy as np
    # 生成一个带有抽象纹理的示例风格图像
    np.random.seed(42)                                    # 固定随机种子以确保可复现
    img = np.zeros((512, 512, 3), dtype=np.uint8)
    # 创建彩色条纹纹理模拟艺术风格
    for i in range(512):
        for j in range(512):
            r = int(127 + 127 * np.sin(i * 0.05 + j * 0.03))   # 红色通道：正弦波
            g = int(127 + 127 * np.sin(i * 0.03 - j * 0.05))   # 绿色通道：不同频率正弦波
            b = int(127 + 127 * np.cos(i * 0.04 + j * 0.04))   # 蓝色通道：余弦波
            img[i, j] = [r, g, b]
    Image.fromarray(img).save(style_image_path)           # 保存为 JPEG 文件
    print(f"示例风格图像已保存到: {style_image_path}")

# 加载图像
content_img = load_image(content_image_path)  # 加载并预处理内容图像
style_img = load_image(style_image_path)      # 加载并预处理风格图像

print(f"\n内容图像形状: {content_img.shape}")   # 应该是 [1, 3, imsize, imsize]
print(f"风格图像形状: {style_img.shape}")

## 第五步：显示输入图像

在开始风格迁移之前，先可视化我们的内容图像和风格图像。

In [ ]:
# 创建一个包含两个子图的画布
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# 显示内容图像
axes[0].imshow(tensor_to_image(content_img))  # 将张量转为图像并显示
axes[0].set_title("内容图像 (Content)", fontsize=14)
axes[0].axis("off")                            # 关闭坐标轴

# 显示风格图像
axes[1].imshow(tensor_to_image(style_img))    # 将张量转为图像并显示
axes[1].set_title("风格图像 (Style)", fontsize=14)
axes[1].axis("off")                            # 关闭坐标轴

plt.tight_layout()  # 自动调整子图间距
plt.show()          # 显示图像

## 第六步：定义内容损失 (Content Loss)

**内容损失**衡量生成图像与内容图像在特征空间中的差异：

$$L_{content} = \frac{1}{2} \sum_{i,j} (F_{ij} - P_{ij})^2$$

其中：
- $F_{ij}$ 是生成图像在某层的特征图
- $P_{ij}$ 是内容图像在同一层的特征图

内容损失确保生成图像保留原始图像的高层语义信息（物体形状、布局等）。

In [ ]:
class ContentLoss(nn.Module):
    """
    内容损失模块。
    计算输入特征图与目标内容特征图之间的均方误差。
    """
    
    def __init__(self, target):
        """
        初始化内容损失。
        
        参数:
            target: 内容图像经过网络某层后的特征图
        """
        super(ContentLoss, self).__init__()
        # 将目标特征图保存下来，detach() 使其不参与梯度计算
        self.target = target.detach()
        # 初始化损失值
        self.loss = 0
    
    def forward(self, input):
        """
        前向传播：计算内容损失。
        
        参数:
            input: 生成图像经过网络某层后的特征图
        返回:
            原样返回输入（损失值保存在 self.loss 中）
        """
        # 计算输入特征与目标特征之间的均方误差
        self.loss = nn.functional.mse_loss(input, self.target)
        # 返回输入，使网络可以继续前向传播
        return input


print("内容损失类定义完成 ✓")

## 第七步：定义风格损失 (Style Loss)

### Gram 矩阵

风格信息通过 **Gram 矩阵** 来表示。Gram 矩阵捕捉了不同特征通道之间的相关性，反映了纹理、色彩等风格信息。

$$G_{ij} = \sum_k F_{ik} \cdot F_{jk}$$

### 风格损失

$$L_{style} = \sum_l w_l \cdot \frac{1}{(2N_lM_l)^2} \sum_{i,j}(G_{ij}^l - A_{ij}^l)^2$$

其中：
- $G^l$ 是生成图像在第 $l$ 层的 Gram 矩阵
- $A^l$ 是风格图像在第 $l$ 层的 Gram 矩阵
- $N_l$ 是第 $l$ 层的特征通道数，$M_l$ 是特征图的空间尺寸

In [ ]:
def gram_matrix(input):
    """
    计算 Gram 矩阵。
    Gram 矩阵是特征图展平后的内积，反映特征通道之间的相关性。
    
    参数:
        input: 特征图张量，形状为 [batch, channels, height, width]
    返回:
        归一化后的 Gram 矩阵，形状为 [batch*channels, batch*channels]
    """
    batch, channels, height, width = input.size()  # 获取张量的各维度大小
    
    # 将特征图展平为 2D 矩阵: [batch*channels, height*width]
    features = input.view(batch * channels, height * width)
    
    # 计算 Gram 矩阵: features × features^T
    G = torch.mm(features, features.t())
    
    # 归一化：除以特征图中的元素总数，避免大特征图主导损失
    G = G.div(batch * channels * height * width)
    
    return G


class StyleLoss(nn.Module):
    """
    风格损失模块。
    通过比较 Gram 矩阵来度量风格差异。
    """
    
    def __init__(self, target_feature):
        """
        初始化风格损失。
        
        参数:
            target_feature: 风格图像在某层的特征图
        """
        super(StyleLoss, self).__init__()
        # 计算风格图像的 Gram 矩阵并保存，detach() 使其不参与梯度计算
        self.target = gram_matrix(target_feature).detach()
        # 初始化损失值
        self.loss = 0
    
    def forward(self, input):
        """
        前向传播：计算风格损失。
        
        参数:
            input: 生成图像在某层的特征图
        返回:
            原样返回输入（损失值保存在 self.loss 中）
        """
        # 计算生成图像的 Gram 矩阵
        G = gram_matrix(input)
        # 计算生成图像与风格图像 Gram 矩阵的均方误差
        self.loss = nn.functional.mse_loss(G, self.target)
        # 返回输入，使网络可以继续前向传播
        return input


print("Gram 矩阵函数和风格损失类定义完成 ✓")

## 第八步：加载预训练 VGG19 模型

VGG19 是一个经过 ImageNet 数据集训练的深层卷积网络。我们使用它的**特征提取部分**（卷积层和池化层），不使用全连接层。

VGG19 在风格迁移中表现优秀，因为：
- 浅层（conv1, conv2）捕获低级特征（边缘、纹理）
- 深层（conv4, conv5）捕获高级特征（物体、语义）

### VGG19 需要的图像标准化
VGG19 训练时使用了特定的均值和标准差进行归一化，我们需要对输入做同样的处理。

In [ ]:
# VGG19 训练时使用的 ImageNet 数据集的均值和标准差
vgg_mean = torch.tensor([0.485, 0.456, 0.406]).to(device)  # RGB 三通道均值
vgg_std = torch.tensor([0.229, 0.224, 0.225]).to(device)   # RGB 三通道标准差


class Normalization(nn.Module):
    """
    图像标准化模块。
    将输入图像按照 VGG19 要求的均值和标准差进行归一化。
    作为模型的第一层使用。
    """
    
    def __init__(self, mean, std):
        """
        参数:
            mean: 各通道均值
            std: 各通道标准差
        """
        super(Normalization, self).__init__()
        # 将均值和标准差变形为 [1, C, 1, 1]，以便与 [B, C, H, W] 的图像进行广播运算
        self.mean = mean.view(-1, 1, 1)
        self.std = std.view(-1, 1, 1)
    
    def forward(self, img):
        """
        对输入图像进行标准化。
        公式: normalized = (img - mean) / std
        """
        return (img - self.mean) / self.std


# 加载预训练的 VGG19 模型，只使用 features 部分（卷积+池化层）
vgg = models.vgg19(weights=models.VGG19_Weights.IMAGENET1K_V1).features.to(device).eval()

# 冻结 VGG19 的所有参数，我们不需要更新它的权重
for param in vgg.parameters():
    param.requires_grad_(False)

print("VGG19 模型加载完成 ✓")
print(f"\nVGG19 特征提取层结构:")
print(vgg)

## 第九步：构建风格迁移模型

我们需要在 VGG19 的特定层之后插入**内容损失层**和**风格损失层**：

| 损失类型 | 使用的 VGG19 层 | 作用 |
|---------|---------------|------|
| 内容损失 | `conv4_2` | 保留图像的高层语义结构 |
| 风格损失 | `conv1_1`, `conv2_1`, `conv3_1`, `conv4_1`, `conv5_1` | 从多个层级捕获风格信息 |

使用多个层的风格损失可以同时捕获**细粒度纹理**（浅层）和**全局风格模式**（深层）。

In [ ]:
# 定义在哪些层插入内容损失和风格损失
# 使用 VGG19 中卷积层的名称，格式为 "conv_层号"
content_layers_default = ['conv_4']           # 内容损失在 conv4_2 后计算
style_layers_default = ['conv_1', 'conv_2', 'conv_3', 'conv_4', 'conv_5']  # 风格损失在5个层后计算


def build_style_transfer_model(vgg, normalization_mean, normalization_std,
                                content_img, style_img,
                                content_layers=content_layers_default,
                                style_layers=style_layers_default):
    """
    构建用于风格迁移的模型。
    在 VGG19 的指定层后插入 ContentLoss 和 StyleLoss 模块。
    
    参数:
        vgg: 预训练 VGG19 的 features 部分
        normalization_mean: VGG19 的归一化均值
        normalization_std: VGG19 的归一化标准差
        content_img: 内容图像张量
        style_img: 风格图像张量
        content_layers: 插入内容损失的层名列表
        style_layers: 插入风格损失的层名列表
    返回:
        model: 构建好的模型
        content_losses: 内容损失模块列表
        style_losses: 风格损失模块列表
    """
    # 创建标准化层作为模型的第一层
    normalization = Normalization(normalization_mean, normalization_std)
    
    # 使用列表收集所有损失模块
    content_losses = []  # 存储内容损失模块的引用
    style_losses = []    # 存储风格损失模块的引用
    
    # 使用 nn.Sequential 构建模型，从标准化层开始
    model = nn.Sequential(normalization)
    
    i = 0  # 卷积层计数器
    
    # 遍历 VGG19 的每一层
    for layer in vgg.children():
        if isinstance(layer, nn.Conv2d):
            # 遇到卷积层，计数器加1
            i += 1
            name = f'conv_{i}'           # 命名为 conv_1, conv_2, ...
        elif isinstance(layer, nn.ReLU):
            name = f'relu_{i}'           # 对应的 ReLU 层
            # 使用非就地(inplace=False)的 ReLU，避免影响 ContentLoss 和 StyleLoss
            layer = nn.ReLU(inplace=False)
        elif isinstance(layer, nn.MaxPool2d):
            name = f'pool_{i}'           # 最大池化层
        elif isinstance(layer, nn.BatchNorm2d):
            name = f'bn_{i}'             # 批归一化层
        else:
            raise RuntimeError(f'未识别的层: {layer.__class__.__name__}')
        
        # 将当前层添加到模型中
        model.add_module(name, layer)
        
        # 如果当前层是需要计算内容损失的层
        if name in content_layers:
            # 将内容图像前向传播到当前层，获取内容特征
            target = model(content_img).detach()
            # 创建内容损失模块并添加到模型中
            content_loss = ContentLoss(target)
            model.add_module(f'content_loss_{i}', content_loss)
            content_losses.append(content_loss)  # 保存引用以便后续获取损失值
        
        # 如果当前层是需要计算风格损失的层
        if name in style_layers:
            # 将风格图像前向传播到当前层，获取风格特征
            target_feature = model(style_img).detach()
            # 创建风格损失模块并添加到模型中
            style_loss = StyleLoss(target_feature)
            model.add_module(f'style_loss_{i}', style_loss)
            style_losses.append(style_loss)  # 保存引用以便后续获取损失值
    
    # 裁剪模型：移除最后一个损失层之后的所有层（它们不影响损失计算）
    for i in range(len(model) - 1, -1, -1):
        if isinstance(model[i], ContentLoss) or isinstance(model[i], StyleLoss):
            break  # 找到最后一个损失层，停止
    
    model = model[:i + 1]  # 只保留到最后一个损失层
    
    return model, content_losses, style_losses


print("模型构建函数定义完成 ✓")

## 第十步：定义优化过程

### 关键概念

在风格迁移中，我们**优化的是图像本身**，而不是网络的权重：
- 初始化：将生成图像初始化为内容图像的副本
- 优化器：使用 **L-BFGS** 优化器（一种拟牛顿法，收敛速度快，适合此场景）
- 损失函数：总损失 = 内容损失 × 内容权重 + 风格损失 × 风格权重

$$L_{total} = \alpha \cdot L_{content} + \beta \cdot L_{style}$$

其中：
- $\alpha$ 是内容权重（通常设为 1）
- $\beta$ 是风格权重（通常设为 $10^5$ ~ $10^6$，因为风格损失值通常远小于内容损失）

In [ ]:
def run_style_transfer(content_img, style_img, input_img,
                       num_steps=300,
                       style_weight=1000000,
                       content_weight=1):
    """
    执行风格迁移的核心函数。
    
    参数:
        content_img: 内容图像张量
        style_img: 风格图像张量
        input_img: 初始生成图像（通常用内容图像拷贝初始化）
        num_steps: 优化迭代次数
        style_weight: 风格损失的权重 (β)
        content_weight: 内容损失的权重 (α)
    返回:
        优化后的生成图像张量
    """
    print("正在构建风格迁移模型...")
    
    # 构建包含损失层的模型
    model, content_losses, style_losses = build_style_transfer_model(
        vgg, vgg_mean, vgg_std, content_img, style_img
    )
    
    # 确保生成图像需要计算梯度（这是我们要优化的目标）
    input_img.requires_grad_(True)
    
    # 将模型设置为评估模式（不需要训练 VGG 的参数）
    model.eval()
    model.requires_grad_(False)
    
    # 使用 L-BFGS 优化器，优化目标是生成图像的像素值
    optimizer = optim.LBFGS([input_img])
    
    print(f"开始优化，共 {num_steps} 步...")
    print(f"内容权重: {content_weight}, 风格权重: {style_weight}")
    print("-" * 60)
    
    # 使用列表包装计数器，以便在闭包中修改
    run = [0]
    
    while run[0] <= num_steps:
        
        def closure():
            """
            L-BFGS 优化器需要的闭包函数。
            每次调用时：
            1. 将像素值裁剪到合法范围
            2. 前向传播计算损失
            3. 反向传播计算梯度
            4. 返回总损失
            """
            # 将生成图像的像素值裁剪到 [0, 1] 范围
            with torch.no_grad():
                input_img.clamp_(0, 1)
            
            # 清零之前的梯度
            optimizer.zero_grad()
            
            # 将生成图像通过模型前向传播（会自动计算各层的损失）
            model(input_img)
            
            # 汇总风格损失
            style_score = 0
            for sl in style_losses:
                style_score += sl.loss         # 累加各层的风格损失
            
            # 汇总内容损失
            content_score = 0
            for cl in content_losses:
                content_score += cl.loss       # 累加各层的内容损失
            
            # 加权求和得到总损失
            style_score *= style_weight         # 乘以风格权重
            content_score *= content_weight     # 乘以内容权重
            total_loss = style_score + content_score  # 总损失
            
            # 反向传播，计算梯度
            total_loss.backward()
            
            # 每 50 步打印一次损失信息
            run[0] += 1
            if run[0] % 50 == 0:
                print(f"第 {run[0]:>4d} 步 | "
                      f"风格损失: {style_score.item():>10.2f} | "
                      f"内容损失: {content_score.item():>10.2f} | "
                      f"总损失: {total_loss.item():>12.2f}")
            
            return total_loss
        
        # L-BFGS 执行一步优化（内部可能多次调用 closure）
        optimizer.step(closure)
    
    # 最终裁剪像素值到合法范围
    with torch.no_grad():
        input_img.clamp_(0, 1)
    
    print("-" * 60)
    print("优化完成！")
    
    return input_img


print("优化函数定义完成 ✓")

## 第十一步：执行风格迁移！

现在万事俱备，让我们运行风格迁移算法：
1. 用内容图像的副本作为初始生成图像
2. 通过迭代优化，逐步将风格图像的纹理融合到生成图像中

> **注意**：此步骤在 CPU 上可能需要几分钟，在 GPU 上通常在 1 分钟内完成。

In [ ]:
# 用内容图像的副本初始化生成图像
# 也可以用随机噪声初始化: input_img = torch.randn_like(content_img)
input_img = content_img.clone()

# 运行风格迁移
output = run_style_transfer(
    content_img,                # 内容图像
    style_img,                  # 风格图像
    input_img,                  # 初始生成图像
    num_steps=300,              # 优化步数（增大可提升效果，但更慢）
    style_weight=1000000,       # 风格权重（增大则更偏向风格）
    content_weight=1            # 内容权重（增大则更偏向内容）
)

## 第十二步：展示最终结果

将内容图像、风格图像和生成图像并排展示，直观对比效果。

In [ ]:
# 创建包含三个子图的画布
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# 显示内容图像
axes[0].imshow(tensor_to_image(content_img))   # 内容图像
axes[0].set_title("内容图像\n(Content Image)", fontsize=13)
axes[0].axis("off")                             # 隐藏坐标轴

# 显示风格图像
axes[1].imshow(tensor_to_image(style_img))     # 风格图像
axes[1].set_title("风格图像\n(Style Image)", fontsize=13)
axes[1].axis("off")

# 显示生成图像（风格迁移结果）
axes[2].imshow(tensor_to_image(output))        # 风格迁移后的生成图像
axes[2].set_title("生成图像\n(Generated Image)", fontsize=13)
axes[2].axis("off")

plt.suptitle("神经风格迁移结果对比", fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()  # 自动调整布局
plt.show()          # 显示所有图像

## 第十三步：保存结果

将生成的图像保存到文件中。

In [ ]:
# 确保输出目录存在
os.makedirs("output", exist_ok=True)

# 将生成图像转为 PIL 格式并保存
output_image = tensor_to_image(output)           # 张量转 PIL 图像
output_path = "output/stylized_result.jpg"       # 输出路径
output_image.save(output_path, quality=95)       # 以 JPEG 格式保存，质量 95%

print(f"生成图像已保存到: {output_path}")
print(f"图像尺寸: {output_image.size}")

## 第十四步：实验 — 调整超参数

你可以通过调整以下参数来获得不同的效果：

| 参数 | 默认值 | 说明 |
|------|--------|------|
| `style_weight` | 1,000,000 | 增大 → 更强的风格化效果 |
| `content_weight` | 1 | 增大 → 更好地保留内容结构 |
| `num_steps` | 300 | 增大 → 更精细的优化，但更慢 |

下面的代码展示了不同风格权重下的效果对比：

In [ ]:
# ==============================================================
# 实验：不同风格权重的效果对比
# 取消下面的注释即可运行（注意：会花费较长时间）
# ==============================================================

# # 要测试的不同风格权重
# style_weights = [100000, 1000000, 10000000]
#
# # 创建子图画布
# fig, axes = plt.subplots(1, len(style_weights), figsize=(6 * len(style_weights), 6))
#
# for idx, sw in enumerate(style_weights):
#     print(f"\n{'='*60}")
#     print(f"测试风格权重: {sw:,}")
#     print(f"{'='*60}")
#
#     # 每次都从内容图像重新开始
#     input_img = content_img.clone()
#
#     # 运行风格迁移
#     result = run_style_transfer(
#         content_img, style_img, input_img,
#         num_steps=200,           # 减少步数以加快速度
#         style_weight=sw,
#         content_weight=1
#     )
#
#     # 显示结果
#     axes[idx].imshow(tensor_to_image(result))
#     axes[idx].set_title(f"风格权重: {sw:,}", fontsize=12)
#     axes[idx].axis("off")
#
# plt.suptitle("不同风格权重的效果对比", fontsize=16, fontweight='bold')
# plt.tight_layout()
# plt.show()

print("提示：取消上方代码的注释并运行，可以对比不同风格权重的效果")

## 总结

### 本项目实现的核心内容

1. **图像预处理**：加载、缩放、标准化
2. **VGG19 特征提取**：利用预训练网络的中间层特征
3. **内容损失**：通过 MSE 保留内容结构
4. **风格损失**：通过 Gram 矩阵匹配纹理风格
5. **L-BFGS 优化**：直接优化图像像素

### 进一步探索方向

- **快速风格迁移**：训练一个前馈网络实现实时风格迁移（Johnson et al., 2016）
- **任意风格迁移**：使用 AdaIN（自适应实例归一化）实现任意风格的实时迁移
- **视频风格迁移**：将风格迁移扩展到视频序列
- **多风格融合**：同时融合多张风格图像的风格

### 参考文献

- Gatys, L.A., Ecker, A.S. and Bethge, M., 2016. *Image style transfer using convolutional neural networks*. CVPR 2016.
- Johnson, J., Alahi, A. and Fei-Fei, L., 2016. *Perceptual losses for real-time style transfer and super-resolution*. ECCV 2016.